# YOLO11n — isolated 704-image cafeteria trial
FLIC **104** + Canteen **175** + PUST **425**. Group-safe split **536 train / 97 val / 71 test**, 3,879 provisional YOLO boxes.
The frozen four-model V1 system is untouched. Metrics are exploratory, **not a verified school field test**.

First select **Runtime → Change runtime type → T4 GPU**. If the full dataset has already been committed under `experiments/yolo-v2-704/yolo_v2_trial_704/` on GitHub, you do not need to upload the ZIP. Otherwise use the original verified archive from the previous chat.


## 1. Clone code and install dependencies


In [ ]:
import subprocess, sys, torch
from pathlib import Path
assert torch.cuda.is_available(), 'Select a Colab GPU runtime and rerun this cell.'
repo = Path('/content/yolo-cafeteria-research')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/cx-57/yolo-cafeteria-research.git', str(repo)], check=True)
exp = repo / 'experiments/yolo-v2-704'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(exp/'requirements.txt')], check=True)
print('CUDA device:', torch.cuda.get_device_name(0))


## 2. Use the dataset from GitHub, or upload the verified original ZIP
If the 704 images and 704 YOLO labels are present in the cloned GitHub repo, use those files directly. Otherwise upload `YOLO_V2_704_GroupSplit_Trial.zip`; its SHA-256 is verified before extraction. Do not substitute the three separate source archives.


In [ ]:
import hashlib, zipfile
data_yaml = exp / 'yolo_v2_trial_704/data.yaml'
if data_yaml.is_file():
    print('Using 704-image dataset cloned directly from GitHub:', data_yaml)
else:
    from google.colab import files
    expected_name = 'YOLO_V2_704_GroupSplit_Trial.zip'
    expected_hash = '16c9990b2cf45a14915ae3344307f4c1bf8dc9dbbd91a31d0b697313a8c21e27'
    uploaded = files.upload()
    assert expected_name in uploaded, f'Upload {expected_name}'
    archive = Path('/content') / expected_name
    if not archive.is_file(): archive.write_bytes(uploaded[expected_name])
    h = hashlib.sha256()
    with archive.open('rb') as src:
        for block in iter(lambda: src.read(1024 * 1024), b''): h.update(block)
    assert h.hexdigest() == expected_hash, 'Wrong ZIP: SHA-256 mismatch'
    with zipfile.ZipFile(archive) as z:
        assert z.testzip() is None, 'Corrupt ZIP member detected'
        assert all(Path(m.filename).parts[:1] == ('yolo_v2_trial_704',) for m in z.infolist()), 'Unexpected ZIP contents'
        z.extractall('/content')
    data_yaml = Path('/content/yolo_v2_trial_704/data.yaml')
assert data_yaml.is_file(), 'Expected grouped dataset YAML not found'
print('Training dataset:', data_yaml)


## 3. Check all 704 images, labels, and group-safe splits
The repo verifier checks image hashes, label hashes, valid boxes, duplicated bytes, group isolation and counts before any training.


In [ ]:
script = exp / 'train_yolo_v2_trial.py'
output = Path('/content/YOLO_V2_trial_results')
command = [sys.executable, str(script), '--data', str(data_yaml), '--output', str(output)]
subprocess.run(command + ['--verify-only'], check=True)


## 4. Train YOLO11n, then run a single held-out test evaluation
Uses the pretrained `yolo11n.pt`, 20 epochs maximum, and validation-based early stopping. No original V1 weights are modified.


In [ ]:
assert not output.exists(), f'Save/remove the existing results folder before starting a new run: {output}'
subprocess.run(command + ['--device', '0', '--batch', '8', '--epochs', '20'], check=True)


## 5. Download trained model and results
Colab runtimes are temporary. Save the trained `best.pt`, test metrics and run configuration immediately after completion.


In [ ]:
import json
summary = output / 'test_summary.json'
assert summary.is_file(), 'Training and held-out test did not finish'
print(json.dumps(json.loads(summary.read_text()), indent=2))
bundle = Path('/content/YOLO_V2_704_trained_results.zip')
keep = ['test_summary.json', 'experiment_config.json', 'resolved_data.yaml',
        'train/weights/best.pt', 'train/results.csv', 'train/args.yaml']
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for rel in keep:
        f = output / rel
        if f.is_file(): z.write(f, arcname=rel)
files.download(str(bundle))
